# Procesamiento básico con `DataFrames`

## Iniciar Spark

In [1]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
9,application_1763396678263_0015,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
sc

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<SparkContext master=yarn appName=livy-session-9>

## Configuración del entorno

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, upper, count, when, avg, sum as spark_sum,
    year, month, dayofmonth, desc, asc, round as spark_round
)
from pyspark.sql.types import IntegerType, StringType

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Cargar datos de S3

In [3]:
bucket = "lmtorresv-datalake"
path = f"s3a://{bucket}/datasets/covid19/Casos_positivos_de_COVID-19_en_Colombia-100K.csv"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
df_raw = spark.read.csv(
    path,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape="\"",
    encoding="UTF-8"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
df_raw.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- fecha reporte web: string (nullable = true)
 |-- ID de caso: integer (nullable = true)
 |-- Fecha de notificación: string (nullable = true)
 |-- Código DIVIPOLA departamento: integer (nullable = true)
 |-- Nombre departamento: string (nullable = true)
 |-- Código DIVIPOLA municipio: integer (nullable = true)
 |-- Nombre municipio: string (nullable = true)
 |-- Edad: integer (nullable = true)
 |-- Unidad de medida de edad: integer (nullable = true)
 |-- Sexo: string (nullable = true)
 |-- Tipo de contagio: string (nullable = true)
 |-- Ubicación del caso: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Código ISO del país: integer (nullable = true)
 |-- Nombre del país: string (nullable = true)
 |-- Recuperado: string (nullable = true)
 |-- Fecha de inicio de síntomas: string (nullable = true)
 |-- Fecha de muerte: string (nullable = true)
 |-- Fecha de diagnóstico: string (nullable = true)
 |-- Fecha de recuperación: string (nullable = true)
 |-- Tipo de r

In [7]:
df_raw.show(1, vertical=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

-RECORD 0-----------------------------------------
 fecha reporte web            | 6/3/2020 0:00:00  
 ID de caso                   | 1                 
 Fecha de notificación        | 2/3/2020 0:00:00  
 Código DIVIPOLA departamento | 11                
 Nombre departamento          | BOGOTA            
 Código DIVIPOLA municipio    | 11001             
 Nombre municipio             | BOGOTA            
 Edad                         | 19                
 Unidad de medida de edad     | 1                 
 Sexo                         | F                 
 Tipo de contagio             | Importado         
 Ubicación del caso           | Casa              
 Estado                       | Leve              
 Código ISO del país          | 380               
 Nombre del país              | ITALIA            
 Recuperado                   | Recuperado        
 Fecha de inicio de síntomas  | 27/2/2020 0:00:00 
 Fecha de muerte              | NULL              
 Fecha de diagnóstico         |

## Borrar y crear columnas

In [8]:
df_raw.columns

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['fecha reporte web', 'ID de caso', 'Fecha de notificación', 'Código DIVIPOLA departamento', 'Nombre departamento', 'Código DIVIPOLA municipio', 'Nombre municipio', 'Edad', 'Unidad de medida de edad', 'Sexo', 'Tipo de contagio', 'Ubicación del caso', 'Estado', 'Código ISO del país', 'Nombre del país', 'Recuperado', 'Fecha de inicio de síntomas', 'Fecha de muerte', 'Fecha de diagnóstico', 'Fecha de recuperación', 'Tipo de recuperación', 'Pertenencia étnica', 'Nombre del grupo étnico']

### Renombrar columnas a nombres más cortos

In [40]:
df = df_raw.toDF(
    "fecha_reporte",
    "id_caso",
    "fecha_notificacion",
    "cod_depto",
    "departamento",
    "cod_municipio",
    "ciudad",
    "edad",
    "unidad_edad",
    "sexo",
    "tipo_contagio",
    "ubicacion",
    "estado",
    "cod_pais",
    "pais",
    "recuperado",
    "fecha_sintomas",
    "fecha_muerte",
    "fecha_diagnostico",
    "fecha_recuperacion",
    "tipo_recuperacion",
    "etnia",
    "grupo_etnico"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame con las nuevas columnas:

In [44]:
df

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[fecha_reporte: string, id_caso: int, fecha_notificacion: string, cod_depto: int, departamento: string, cod_municipio: int, ciudad: string, edad: int, unidad_edad: int, sexo: string, tipo_contagio: string, ubicacion: string, estado: string, cod_pais: int, pais: string, recuperado: string, fecha_sintomas: string, fecha_muerte: string, fecha_diagnostico: string, fecha_recuperacion: string, tipo_recuperacion: string, etnia: int, grupo_etnico: string]

In [45]:
# Normalizar texto a mayúsculas
df = df.withColumn("departamento", upper(col("departamento")))
df = df.withColumn("ciudad", upper(col("ciudad")))
df = df.withColumn("sexo", upper(col("sexo")))
df = df.withColumn("estado", upper(col("estado")))
df = df.withColumn("ubicacion", upper(col("ubicacion")))
df = df.withColumn("recuperado", upper(col("recuperado")))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [46]:
df.show(1, vertical=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

-RECORD 0-------------------------------
 fecha_reporte      | 6/3/2020 0:00:00  
 id_caso            | 1                 
 fecha_notificacion | 2/3/2020 0:00:00  
 cod_depto          | 11                
 departamento       | BOGOTA            
 cod_municipio      | 11001             
 ciudad             | BOGOTA            
 edad               | 19                
 unidad_edad        | 1                 
 sexo               | F                 
 tipo_contagio      | Importado         
 ubicacion          | CASA              
 estado             | LEVE              
 cod_pais           | 380               
 pais               | ITALIA            
 recuperado         | RECUPERADO        
 fecha_sintomas     | 27/2/2020 0:00:00 
 fecha_muerte       | NULL              
 fecha_diagnostico  | 6/3/2020 0:00:00  
 fecha_recuperacion | 13/3/2020 0:00:00 
 tipo_recuperacion  | PCR               
 etnia              | 6                 
 grupo_etnico       | NULL              
only showing top

Hacer caché del DataFrame:

In [47]:
df.cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[fecha_reporte: string, id_caso: int, fecha_notificacion: string, cod_depto: int, departamento: string, cod_municipio: int, ciudad: string, edad: int, unidad_edad: int, sexo: string, tipo_contagio: string, ubicacion: string, estado: string, cod_pais: int, pais: string, recuperado: string, fecha_sintomas: string, fecha_muerte: string, fecha_diagnostico: string, fecha_recuperacion: string, tipo_recuperacion: string, etnia: int, grupo_etnico: string]

## Aplicar filtros para el análisis

In [69]:
# Casos leves
activos = df.filter(col("estado") == "LEVE")
activos.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

94367

In [70]:
# Fallecimientos
fallecimientos = df.filter(col("estado") == "FALLECIDO")
fallecimientos.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

4663

In [71]:
# Recuperados
recuperados = df.filter(col("recuperado") == "RECUPERADO")
recuperados.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

94832

## Agrupaciones y consultas categóricas

In [72]:
# Casos por sexo
casos_por_sexo = df.groupBy("sexo").count().orderBy(desc("count"))
casos_por_sexo.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+-----+
|sexo|count|
+----+-----+
|   M|54098|
|   F|45902|
+----+-----+

In [73]:
# Casos por tipo de contagio
casos_por_tipo_de_contagio = df.groupBy("tipo_contagio").count().orderBy(desc("count"))
casos_por_tipo_de_contagio.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------+-----+
|tipo_contagio|count|
+-------------+-----+
|  Comunitaria|58699|
|  Relacionado|40084|
|    Importado|  912|
|   En estudio|  305|
+-------------+-----+

In [74]:
# Casos por ubicación
casos_por_ubicacion = df.groupBy("ubicacion").count().orderBy(desc("count"))
casos_por_ubicacion.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+-----+
|ubicacion|count|
+---------+-----+
|     CASA|94367|
|FALLECIDO| 4663|
|      N/A|  970|
+---------+-----+

In [75]:
# Distribución por estado
distribucion_por_estado = df.groupBy("estado").count().orderBy(desc("count"))
distribucion_por_estado.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+-----+
|   estado|count|
+---------+-----+
|     LEVE|94367|
|FALLECIDO| 4663|
|      N/A|  970|
+---------+-----+

In [76]:
# Casos por tipo de recuperación
casos_por_tipo_de_recuperacion = df.groupBy("tipo_recuperacion").count().orderBy(desc("count"))
casos_por_tipo_de_recuperacion

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[tipo_recuperacion: string, count: bigint]

In [82]:
from pyspark.sql.functions import when

# Mapear códigos de etnia a nombres descriptivos
df = df.withColumn(
    "etnia_desc",
    when(col("etnia") == "1", "Indígena")
    .when(col("etnia") == "2", "ROM (Gitano)")
    .when(col("etnia") == "3", "Raizal")
    .when(col("etnia") == "4", "Palenquero")
    .when(col("etnia") == "5", "Negro/Afrodescendiente")
    .when(col("etnia") == "6", "Otro")
    .otherwise("No reportado")
)

# Casos por pertenencia étnica
casos_por_pertenencia_etnica = df.groupBy("etnia", "etnia_desc").count().orderBy(desc("count"))
casos_por_pertenencia_etnica.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+--------------------+-----+
|etnia|          etnia_desc|count|
+-----+--------------------+-----+
|    6|                Otro|80809|
|    5|Negro/Afrodescend...|13402|
|    1|            Ind?gena| 5768|
|    3|              Raizal|   19|
|    2|        ROM (Gitano)|    2|
+-----+--------------------+-----+

## Guardar resultados

In [78]:
output_bucket = "lmtorresv-datalake"
output_base = f"s3a://{output_bucket}/covid/resultados/"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [79]:
def guardar_resultado(dataframe, nombre):
    dataframe.coalesce(1) \
        .write.mode("overwrite") \
        .option("header", True) \
        .csv(output_base + nombre)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [80]:
guardar_resultado(activos, "activos")
guardar_resultado(fallecimientos, "fallecimientos")
guardar_resultado(recuperados, "recuperados")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [81]:
guardar_resultado(casos_por_sexo, "casos_por_sexo")
guardar_resultado(casos_por_tipo_de_contagio, "casos_por_tipo_de_contagio")
guardar_resultado(casos_por_ubicacion, "casos_por_ubicacion")
guardar_resultado(distribucion_por_estado, "distribucion_por_estado")
guardar_resultado(casos_por_tipo_de_recuperacion, "casos_por_tipo_de_recuperacion")
guardar_resultado(casos_por_pertenencia_etnica, "casos_por_pertenencia_etnica")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [83]:
df.unpersist()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[fecha_reporte: string, id_caso: int, fecha_notificacion: string, cod_depto: int, departamento: string, cod_municipio: int, ciudad: string, edad: int, unidad_edad: int, sexo: string, tipo_contagio: string, ubicacion: string, estado: string, cod_pais: int, pais: string, recuperado: string, fecha_sintomas: string, fecha_muerte: string, fecha_diagnostico: string, fecha_recuperacion: string, tipo_recuperacion: string, etnia: int, grupo_etnico: string, etnia_desc: string]